# Class 33: Image Fitting: Point Sources and Photometry
## Objective: Understand imaging data and photometric measurements

These exercises are based on those in "Lecture 20: Image Fitting: Point Sources and Photometry" by Yuan-Sen Ting and available from https://tingyuansen.github.io/coding_essential_for_astronomers/lectures/lecture20-image-fitting-point-sources-photometry.html

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from scipy import ndimage
from scipy import stats
from astropy import units as u
from astropy import constants as const

# Set random seed for reproducibility
np.random.seed(52)

# Configure matplotlib for better-looking plots
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

## Section 1: Angular Resolution, the Diffraction Limit, and Atmospheric Turbulence (or Seeing)

The angular size of an object (e.g. a star of radius $R_*$) is:

$$
\theta \approx \frac{2 R}{d}
$$
This is based on the small-angle approximation, which is an extremely good approximation for objects at astronomical distances. Note the units are radians. 

It may be helpful to remember that there are 206265 arcseconds in a radian. 

The diffraction limit of an optical system is the minimum resolvable angular size. The most common formula is the Rayleigh Criterion:

$$
\theta = 1.22 \frac{\lambda}{D}
$$

The diffraction limit improves (gets smaller) linearly as the diameter of the optical system increases, and degrades linearly as the wavelength of light increases. 

The diffraction limit is the best case scenario, and most optical systems are not diffraction-limited because they are not perfect (including the human eye). 

In [ ]:
# Calculate the angular size of a few objects

# Sun observed from Earth
R_sun = const.R_sun  # Solar radius
d_earth = 1 * u.AU
theta_from_earth = (2 * R_sun / d_earth).to(u.arcsec, 
                                                 equivalencies=u.dimensionless_angles())

print(f"Sun from Earth: {theta_from_earth:.6f} or {theta_from_earth.to(u.deg):.4f}")

# Sun from Alpha Centauri (nearest star system, 4.37 light-years)
d_alpha_cen = 4.37 * u.lightyear
theta_alpha_cen = (2 * R_sun / d_alpha_cen).to(u.arcsec, 
                                                 equivalencies=u.dimensionless_angles())

print(f"Sun from Alpha Centauri: {theta_alpha_cen:.6f}")

# Calculate the diffraction limit of a few optical systems

# Human eye
d_eye = 8 * u.mm # fully dilated human pupil in mm
lambda_vis = 550 * u.nm
theta_eye = (1.22 * lambda_vis / d_eye).to(u.arcsec, equivalencies=u.dimensionless_angles())

print(f"Angular resolution of the human eye: {theta_eye:.4f}") 

# HST
d_hst = 2.5 * u.m # 
lambda_vis = 550 * u.nm
theta_hst = (1.22 * lambda_vis / d_hst).to(u.arcsec, equivalencies=u.dimensionless_angles())

print(f"Angular resolution of HST: {theta_hst:.4f}") 

# GBT
d_gbt = 100 * u.m # 
lambda_hi = 21 * u.cm
theta_gbt = (1.22 * lambda_hi / d_gbt).to(u.arcsec, equivalencies=u.dimensionless_angles())

print(f"Angular resolution of GBT at 21 cm: {theta_gbt:.4f} or {theta_gbt.to(u.arcmin):.4f}") 

**Test your understanding:** 
Calculate the angular resolution of JWST (D = 6.5-m) at 1.5 microns. Compare this to the angular resolution of HST at 550nm. 

## Section 2: The Point Spread Function

The **point spread function** or PSF is a function that represents the flux distribution of a point-like or unresolved object. 

A 2D Gaussian is a good approximation to a PSF:

$$
PSF(x,y) = A \exp \left( - \frac{ (x - x_0)^2 + (y - y_0)^2}{2 \sigma^2} \right) 
$$

Astronomers commonly use the **full width at half maximum** (FWHM) to characterize the width of the PSF. For a Gaussian profile, the FWHM is related to the standard deviation $\sigma$ by 

$$
\mathrm{FWHM} \approx 2.355 \sigma
$$


In [ ]:
# Create a coordinate grid
size = 50
x = np.arange(size)
y = np.arange(size)
x_grid, y_grid = np.meshgrid(x, y)

# PSF parameters
x0, y0 = 25, 25
amplitude = 100
sigma = 3.0

# Calculate 2D Gaussian
r_squared = (x_grid - x0)**2 + (y_grid - y0)**2
psf = amplitude * np.exp(-r_squared / (2 * sigma**2))
fwhm = 2.355 * sigma

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 2D image
im1 = axes[0].imshow(psf, cmap='hot', origin='lower', interpolation='nearest')
axes[0].set_title(f'Gaussian PSF (FWHM = {fwhm:.1f} pixels)', 
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('X [pixels]')
axes[0].set_ylabel('Y [pixels]')
plt.colorbar(im1, ax=axes[0], label='Intensity')

# Cross-section through center
center_slice = size // 2
axes[1].plot(x, psf[center_slice, :], 'b-', lw=2)
axes[1].axhline(amplitude / 2, color='red', ls='--', lw=1, alpha=0.5)
axes[1].axvline(x0 - fwhm/2, color='red', ls='--', lw=1, alpha=0.5)
axes[1].axvline(x0 + fwhm/2, color='red', ls='--', lw=1, alpha=0.5)
axes[1].set_xlabel('X [pixels]', fontsize=11)
axes[1].set_ylabel('Intensity', fontsize=11)
axes[1].set_title('Cross-Section', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(x0, amplitude*0.55, 'FWHM', ha='center', fontsize=10, color='red')

# Radial profile (log scale shows wings)
radius = np.sqrt((x_grid - x0)**2 + (y_grid - y0)**2).flatten()
intensity = psf.flatten()
sort_idx = np.argsort(radius)

axes[2].plot(radius[sort_idx], intensity[sort_idx], 'b-', lw=2, alpha=0.7)
axes[2].set_xlabel('Radius [pixels]', fontsize=11)
axes[2].set_ylabel('Intensity (log scale)', fontsize=11)
axes[2].set_title('Radial Profile', fontsize=12, fontweight='bold')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3, which='both')
axes[2].set_xlim(0, 25)

plt.tight_layout()
plt.show()

**Test your understanding:** If a telescope operator tells you the "seeing" tonight is 1.2 arcseconds (FWHM), what value of $\sigma$ should you use if you are trying to model the stars with a Gaussian function?

In [ ]:
fwhm_seeing = 1.2 # arcseconds

# Calculate sigma
# Enter your code here

# Section 3: CCD Calibration, Noise, and Signal-to-Noise Ratio (SNR)

Key words are:

**Quantum Efficiency:** Describes the fraction of incoming electrons that are successfully recorded by the detector as electrons. 

**Gain:** is the conversion of these electrons into **Analog-to-Digital Units** or **ADUs**. These are what are typically recorded in fits images from telescopes. 

**Poisson Noise:** Uncertainty in the number of photons hitting the pixel ($\sigma = \sqrt{N}$).

**Read Noise:** A constant electronic noise added by the camera hardware, regardless of how much light there is.

**Signal-to-Noise (SNR):** Literally the ratio of the signal to the noise. This is a standard measure of the significance of a detection. 

**Background noise:** Pervasive noise in the background of an astronomical image. 

In [ ]:
# Gain: how many electrons equal one "count" (ADU)
gain = 2.0 # electrons/ADU

# Measured flux in ADU
measured_adu = 1000.
electrons = measured_adu * gain

# Poisson noise is calculated in ELECTRONS (physical particles)
electron_noise = np.sqrt(electrons)

# Convert noise back to ADU
adu_noise = electron_noise / gain

print(f"Measured: {measured_adu:.0f} ADU | Noise: {adu_noise:.2f} ADU")
print(f"Measured: {electrons:.0f} electrons | Noise: {electron_noise:.2f} electrons")

**Test your understanding:** You measure a star with 5000 ADU using a camera with a gain of 1.5. Calculate the Poisson noise in ADU.

In [ ]:
# Enter your code here

adu = 5000
gain = 1.5

# Step 1: Convert to electrons
# Step 2: Find sqrt(electrons)
# Step 3: Convert back to ADU

In [ ]:
# Create a synthetic image

# Image parameters
image_size = 200  # pixels
n_stars = 5  # number of stars
background_level = 100.0  # counts per pixel
read_noise = 10.0  # electrons per pixel
psf_sigma = 2.5  # pixels (FWHM ~ 6 pixels, space-quality)

# Star positions (margin to keep away from edges)
margin = 15
np.random.seed(522)
star_x = np.random.uniform(margin, image_size - margin, n_stars)
star_y = np.random.uniform(margin, image_size - margin, n_stars)

# Star brightnesses (log-normal distribution)
mean_log_brightness = np.log(800)
std_log_brightness = 0.6
star_amplitudes = np.exp(np.random.normal(mean_log_brightness, std_log_brightness, n_stars))

print(f"Star amplitudes: {star_amplitudes.min():.0f} to {star_amplitudes.max():.0f} counts")
print(f"Median amplitude: {np.median(star_amplitudes):.0f} counts")

# Create coordinate grids
x_im = np.arange(image_size)
y_im = np.arange(image_size)
x_grid_im, y_grid_im = np.meshgrid(x_im, y_im)

# Initialize with background
image = np.full((image_size, image_size), background_level)

# Add each star
for i in range(n_stars):
    r_squared = (x_grid_im - star_x[i])**2 + (y_grid_im - star_y[i])**2
    star_psf = star_amplitudes[i] * np.exp(-r_squared / (2 * psf_sigma**2))
    image += star_psf

# Add Poisson noise (photon statistics)
poisson_noise = np.random.poisson(image).astype(float) - image

# Add read noise (Gaussian, from detector electronics)
read_noise_array = np.random.normal(0, read_noise, image.shape)

# Final noisy image
image_noisy = image + poisson_noise + read_noise_array
image_noisy = np.maximum(image_noisy, 0)  # No negative values  

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# True noiseless image
im1 = axes[0].imshow(image, cmap='gray', origin='lower', interpolation='nearest',
                     vmin=background_level - 50, vmax=background_level + 500)
axes[0].scatter(star_x, star_y, c='red', s=50, marker='x', alpha=0.7, linewidths=2)
axes[0].set_title('True Image (No Noise)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('X [pixels]')
axes[0].set_ylabel('Y [pixels]')
plt.colorbar(im1, ax=axes[0], label='Counts')

# Noisy image (realistic)
im2 = axes[1].imshow(image_noisy, cmap='gray', origin='lower', interpolation='nearest',
                     vmin=background_level - 50, vmax=background_level + 500)
axes[1].set_title('Noisy Image (Realistic)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('X [pixels]')
axes[1].set_ylabel('Y [pixels]')
plt.colorbar(im2, ax=axes[1], label='Counts')

# Zoom into a region
zoom_size = 40
zoom_x, zoom_y = 80, 80
im3 = axes[2].imshow(image_noisy[zoom_y:zoom_y+zoom_size, zoom_x:zoom_x+zoom_size], 
                     cmap='gray', origin='lower', interpolation='nearest',
                     vmin=background_level - 50, vmax=background_level + 500)
axes[2].set_title('Zoomed Region', fontsize=12, fontweight='bold')
axes[2].set_xlabel('X [pixels]')
axes[2].set_ylabel('Y [pixels]')
plt.colorbar(im3, ax=axes[2], label='Counts')

plt.tight_layout()
plt.show()

## Section 4: Background Estimation

Images contain more than just starlight; they include "sky background" from the atmosphere and telescope glow. Before we measure a star, we must subtract this background. A common method is to use an annulus (a ring) around the star to estimate the "empty" sky level using the median value, which is resistant to outlier pixels or faint neighboring stars.

Note that removing this background does not remove the noise that it produced. It is important to still account for this noise contribution.

In [ ]:
# Simple median (robust to stars which are outliers)
background_median = np.median(image_noisy)

print(f"True Background: {background_level} counts/pixel")
print(f"Median Background: {background_median:.2f} counts/pixel")

# Estimate the background with sigma-clipped statistics
clipped_data = image_noisy.flatten().copy()

for iteration in range(5):
    mean = np.mean(clipped_data)
    std = np.std(clipped_data)
    # Keep only pixels within 3σ of mean
    mask = np.abs(clipped_data - mean) < 3 * std
    clipped_data = clipped_data[mask]

bg_mean = np.mean(clipped_data)
bg_std = np.std(clipped_data)

print(f"Sigma-clipped Background: {bg_mean:.2f} counts/pixel")
print(f"Background noise (std): {bg_std:.2f} counts/pixel")

In [ ]:
# Illustration of background subtraction
image_bgsub = image_noisy - bg_mean

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Original
im1 = axes[0].imshow(image_noisy, cmap='gray', origin='lower', interpolation='nearest',
                     vmin=50, vmax=600)
axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0].set_xlabel('X [pixels]')
axes[0].set_ylabel('Y [pixels]')
plt.colorbar(im1, ax=axes[0], label='Counts')

# Background-subtracted
im2 = axes[1].imshow(image_bgsub, cmap='gray', origin='lower', interpolation='nearest',
                     vmin=-50, vmax=500)
axes[1].set_title('Background-Subtracted', fontsize=12, fontweight='bold')
axes[1].set_xlabel('X [pixels]')
axes[1].set_ylabel('Y [pixels]')
plt.colorbar(im2, ax=axes[1], label='Counts')

# Histogram comparison
axes[2].hist(image_noisy.flatten(), bins=50, alpha=0.6, label='Original', 
             color='blue', range=(0, 400))
axes[2].hist(image_bgsub.flatten(), bins=50, alpha=0.6, label='Background-subtracted', 
             color='red', range=(-100, 300))
axes[2].axvline(0, color='black', ls='--', lw=1)
axes[2].set_xlabel('Pixel Value [counts]', fontsize=11)
axes[2].set_ylabel('Number of Pixels', fontsize=11)
axes[2].set_title('Pixel Distributions', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Test your understanding:** A student calculates the background using the mean (average) of a region that accidentally includes a very bright star. Another student uses the median. Which student will have a background estimate that is too high, and why? Provide a code snippet comparing the mean and median of the array data provided below.

In [ ]:
data = np.array([50, 52, 49, 51, 48, 5000]) # The 5000 is a bright star pixel

# Calculate mean_val and median_val

## Section 5: Source Detection

It is very valuable to automatically detect sources in an image. This is easier to do after background subtraction. 

The key point is that a source is an object that is brighter than the background level. Then any pixel (or small region of pixels) that is a substantial outlier could be classified as a source. The function `scipy.nimage.maximum_filter` is one way to do that.

The following code identifies sources that are more than $5\sigma$ brighter than the background. 

In [ ]:
# Detection threshold
detection_threshold = 5 * bg_std

print(f"Detection threshold: {detection_threshold:.2f} counts (5σ)")

# Find local maxima
filter_size = int(np.ceil(2.355 * psf_sigma))  # ~FWHM

local_max = ndimage.maximum_filter(image_bgsub, size=filter_size)

# A pixel is a peak if it equals the local maximum AND exceeds threshold
is_peak = (image_bgsub == local_max) & (image_bgsub > detection_threshold)

# Get coordinates
detected_y, detected_x = np.where(is_peak)
n_detected = len(detected_x)

print(f"Detected {n_detected} sources (true number: {n_stars})")

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

im = ax.imshow(image_bgsub, cmap='gray', origin='lower', interpolation='nearest',
               vmin=-50, vmax=500)
ax.scatter(detected_x, detected_y, c='red', s=200, marker='o', 
          facecolors='none', edgecolors='red', linewidths=2, label=f'Detected (n={n_detected})')
ax.scatter(star_x, star_y, c='cyan', s=50, marker='x', 
          linewidths=2, label='True positions', alpha=0.7)
ax.set_title('Source Detection', fontsize=13, fontweight='bold')
ax.set_xlabel('X [pixels]')
ax.set_ylabel('Y [pixels]')
ax.legend()
plt.colorbar(im, ax=ax, label='Counts')

plt.tight_layout()
plt.show()

## Section 6: Aperture Photometry

Aperture photometry is the process of summing up all the pixels in a circle (the aperture) centered on a star and then subtracting the background.

$$
F_{star} = \sum_i^{aperture} f_i - N_{ap} \times < sky >
$$

There are important considerations in the choice of aperture size: 
- **Small Aperture:** Misses starlight (low "Encircled Energy") but has less background noise.
- **Large Aperture:** Captures all starlight but includes more noisy background pixels.

The optimal choice of aperture size depends on factors like how bright the source is relative to the background and how crowded the field is. Crowding occurs when a second source significantly contributes to the flux measured in an aperture.

Even if we have subtracted the background flux, it is usually helpful to compute the local background based on the nearest pixels. This is usually performed with a *sky annulus* and captures local variations in the sky background. 

In [ ]:
# Assume we have a 10x10 cutout of a star
star_cutout = np.array([[10, 10, 10], [10, 100, 10], [10, 10, 10]]) 
bg_per_pixel = 5

# Sum the pixels
total_sum = np.sum(star_cutout)

# Subtract background: Sum - (number_of_pixels * bg_level)
n_pixels = star_cutout.size
net_flux = total_sum - (n_pixels * bg_per_pixel)

print(f"Net Star Flux: {net_flux}")

In [ ]:
# Visualization of aperture photometry

# Aperture sizes
fwhm = 2.355 * psf_sigma
aperture_radius = 2.5 * fwhm
sky_inner_radius = 3.0 * fwhm
sky_outer_radius = 4.0 * fwhm

print(f"FWHM: {fwhm:.2f} pixels")
print(f"Aperture radius: {aperture_radius:.2f} pixels")
print(f"Sky annulus: {sky_inner_radius:.2f} to {sky_outer_radius:.2f} pixels")

# Measure photometry for the brightest detected star
brightest_idx = 0  # We'll update this after measuring all stars

# For now, measure the first star
x_center = detected_x[0]
y_center = detected_y[0]

# Create distance array
y_grid, x_grid = np.ogrid[0:image_size, 0:image_size]
r = np.sqrt((x_grid - x_center)**2 + (y_grid - y_center)**2)

# Define masks
aperture_mask = (r <= aperture_radius)
sky_mask = (r >= sky_inner_radius) & (r <= sky_outer_radius)

# Sky background
sky_pixels = image_bgsub[sky_mask]
sky_median = np.median(sky_pixels)

# Source flux
aperture_pixels = image_bgsub[aperture_mask]
n_aperture_pixels = len(aperture_pixels)
flux = np.sum(aperture_pixels) - n_aperture_pixels * sky_median

# Measure all detected sources
aperture_fluxes = []

for i in range(n_detected):
    x_c, y_c = detected_x[i], detected_y[i]
    r = np.sqrt((x_grid - x_c)**2 + (y_grid - y_c)**2)
    
    # Masks
    ap_mask = (r <= aperture_radius)
    sky_mask = (r >= sky_inner_radius) & (r <= sky_outer_radius)
    
    # Sky
    sky_pix = image_bgsub[sky_mask]
    if len(sky_pix) > 0:
        sky_med = np.median(sky_pix)
    else:
        sky_med = 0.0
    
    # Flux
    ap_pix = image_bgsub[ap_mask]
    flux = np.sum(ap_pix) - len(ap_pix) * sky_med
    aperture_fluxes.append(flux)

aperture_fluxes = np.array(aperture_fluxes)

# Show results for brightest stars
brightest_5 = np.argsort(aperture_fluxes)[-5:][::-1]
print(f"\nAperture Photometry Results (5 brightest):")
print(f"{'Rank':<6} {'Flux (counts)':<15}")
for i, idx in enumerate(brightest_5, 1):
    print(f"{i:<6} {aperture_fluxes[idx]:<15.0f}")

fig, ax = plt.subplots(1, 1, figsize=(10, 10))

im = ax.imshow(image_bgsub, cmap='gray', origin='lower', interpolation='nearest',
               vmin=-50, vmax=500)

# Plot apertures for 5 brightest stars
brightest_indices = np.argsort(aperture_fluxes)[-5:]

for idx in brightest_indices:
    x, y = detected_x[idx], detected_y[idx]
    
    # Source aperture
    circle_ap = plt.Circle((x, y), aperture_radius, 
                          color='red', fill=False, lw=2)
    ax.add_patch(circle_ap)
    
    # Sky annulus
    circle_in = plt.Circle((x, y), sky_inner_radius,
                          color='cyan', fill=False, lw=1.5, ls='--')
    circle_out = plt.Circle((x, y), sky_outer_radius,
                           color='cyan', fill=False, lw=1.5, ls='--')
    ax.add_patch(circle_in)
    ax.add_patch(circle_out)

ax.set_title('Aperture Photometry (5 Brightest Stars)', fontsize=13, fontweight='bold')
ax.set_xlabel('X [pixels]')
ax.set_ylabel('Y [pixels]')
plt.colorbar(im, ax=ax, label='Counts')

plt.tight_layout()
plt.show()

**Test your understanding:** You have a circular aperture containing 50 pixels. The total sum of counts in those pixels is 8500 ADU. The background level was measured to be 20 ADU per pixel. What is the net flux of the star?

In [ ]:
total_counts = 8500
n_pix = 50
bg_level = 20

# Calculate net_flux

## Section 7: PSF Fitting

When stars are crowded or we need high precision, we can use `scipy.optimize.curve_fit` to model the pixels as a 2D Gaussian. 

Here is an example of fitting a model PSF to a single source. Note that because `curve_fit` only accepts 1D arrays, we must flatten our 2D image and coordinates.

In [ ]:
# Select the brightest detected star
brightest_idx = np.argmax(aperture_fluxes)
x_init = detected_x[brightest_idx]
y_init = detected_y[brightest_idx]

# Extract small region around star
box_size = int(4 * fwhm)
half_box = box_size // 2

x_min = max(0, int(x_init) - half_box)
x_max = min(image_size, int(x_init) + half_box)
y_min = max(0, int(y_init) - half_box)
y_max = min(image_size, int(y_init) + half_box)

cutout = image_bgsub[y_min:y_max, x_min:x_max]

# Use peak detection position directly as initial guess
x_initial = x_init
y_initial = y_init

# Use peak detection position directly
print(f"Initial detection position: ({x_init:.1f}, {y_init:.1f})")
print(f"This position will be refined by the PSF fitting algorithm")

# Prepare cutout coordinates
cutout_shape = cutout.shape
x_cutout = np.arange(cutout_shape[1]) + x_min
y_cutout = np.arange(cutout_shape[0]) + y_min
x_grid_cut, y_grid_cut = np.meshgrid(x_cutout, y_cutout)

# Flatten for curve_fit
x_flat = x_grid_cut.ravel()
y_flat = y_grid_cut.ravel()
data_flat = cutout.ravel()

def gaussian_psf_model(xy, x0, y0, amplitude, sigma, background):
    """2D Gaussian PSF for curve_fit (takes and returns flattened arrays)"""
    x, y = xy
    r_squared = (x - x0)**2 + (y - y0)**2
    model = amplitude * np.exp(-r_squared / (2 * sigma**2)) + background
    return model

# Initial guesses
p0 = [
    x_initial,  # x0: use peak detection position
    y_initial,  # y0
    aperture_fluxes[brightest_idx] / (2 * np.pi * psf_sigma**2),  # amplitude (rough estimate)
    psf_sigma,  # sigma
    0.0  # background (should be ~0 for background-subtracted image)
]

# Fit the model
popt, pcov = optimize.curve_fit(
    gaussian_psf_model,
    (x_flat, y_flat),
    data_flat,
    p0=p0
)

# Extract fitted parameters
x0_fit, y0_fit, amp_fit, sigma_fit, bg_fit = popt
perr = np.sqrt(np.diag(pcov))  # Parameter uncertainties

# Total flux from fitted Gaussian (integral = 2π σ² A)
flux_psf = 2 * np.pi * sigma_fit**2 * amp_fit
flux_psf_err = flux_psf * np.sqrt((2*perr[3]/sigma_fit)**2 + (perr[2]/amp_fit)**2)

# Model
model_flat = gaussian_psf_model((x_flat, y_flat), *popt)
model_cut = model_flat.reshape(cutout_shape)

# Residuals
residual_cut = cutout - model_cut

# Pixel uncertainties (approximate as sqrt of counts for Poisson noise)
sigma_pixels = np.sqrt(np.abs(data_flat) + bg_std**2)

# Chi-squared
chi_squared = np.sum((data_flat - model_flat)**2 / sigma_pixels**2)

# Degrees of freedom
n_data = len(data_flat)
n_params = 5  # x0, y0, amplitude, sigma, background
dof = n_data - n_params

# Reduced chi-squared
chi_squared_red = chi_squared / dof

print(f"Chi-squared: {chi_squared:.2f}")
print(f"Reduced chi-squared: {chi_squared_red:.2f}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 11))

# Data
im1 = axes[0, 0].imshow(cutout, cmap='hot', origin='lower', interpolation='nearest')
axes[0, 0].plot(x_initial - x_min, y_initial - y_min, 'c+', markersize=15, markeredgewidth=2)
axes[0, 0].set_title('Data', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('X [pixels]')
axes[0, 0].set_ylabel('Y [pixels]')
plt.colorbar(im1, ax=axes[0, 0], label='Counts')

# Model
im2 = axes[0, 1].imshow(model_cut, cmap='hot', origin='lower', interpolation='nearest')
axes[0, 1].plot(x0_fit - x_min, y0_fit - y_min, 'c+', markersize=15, markeredgewidth=2)
axes[0, 1].set_title('Fitted Model', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('X [pixels]')
axes[0, 1].set_ylabel('Y [pixels]')
plt.colorbar(im2, ax=axes[0, 1], label='Counts')

# Residuals
vmax_res = max(abs(residual_cut.min()), abs(residual_cut.max()))
im3 = axes[1, 0].imshow(residual_cut, cmap='RdBu_r', origin='lower',
                       interpolation='nearest', vmin=-vmax_res, vmax=vmax_res)
axes[1, 0].set_title(f'Residuals\nχ²_red = {chi_squared_red:.2f}', 
                    fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('X [pixels]')
axes[1, 0].set_ylabel('Y [pixels]')
plt.colorbar(im3, ax=axes[1, 0], label='Counts')

# Radial profile
r_cut = np.sqrt((x_grid_cut - x0_fit)**2 + (y_grid_cut - y0_fit)**2).ravel()
sort_idx = np.argsort(r_cut)

axes[1, 1].plot(r_cut[sort_idx], data_flat[sort_idx], 'b.', 
               alpha=0.3, markersize=3, label='Data')
axes[1, 1].plot(r_cut[sort_idx], model_flat[sort_idx], 'r-', lw=2, label='Model')
axes[1, 1].set_xlabel('Radius [pixels]', fontsize=11)
axes[1, 1].set_ylabel('Intensity [counts]', fontsize=11)
axes[1, 1].set_title('Radial Profile', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Test your understanding:** You are fitting a star located at (x=12,y=15). Your initial_guess for the parameters [$x_0$ , $y_0$,amplitude, $\sigma$] is [10,10,500,2]. Why might providing an initial guess of [100,100,500,2] cause the fit to fail?

**Answer in a comment:**


### Section 8: Crowded Fields: Fitting Multiple Sources

PSF fitting is really valuable if we have multiple sources close together such that aperture photometry will not work. That is, if sources are too close together, the aperture for a given object will be contaminated by other, nearby objects. 

In [ ]:
# Create crowded synthetic scene
crowded_size = 60
n_crowd = 3

# Positions (some close together)
np.random.seed(523)
crowd_x = [18, 25, 42]
crowd_y = [20, 30, 25]

# Amplitudes
crowd_amps = [1200, 900, 1000]

# Generate crowded image
x_c = np.arange(crowded_size)
y_c = np.arange(crowded_size)
x_grid_c, y_grid_c = np.meshgrid(x_c, y_c)

crowd_image = np.full((crowded_size, crowded_size), background_level)

for i in range(n_crowd):
    r_sq = (x_grid_c - crowd_x[i])**2 + (y_grid_c - crowd_y[i])**2
    crowd_image += crowd_amps[i] * np.exp(-r_sq / (2 * psf_sigma**2))

# Add noise
crowd_noisy = (np.random.poisson(crowd_image).astype(float) +
               np.random.normal(0, read_noise, crowd_image.shape))
crowd_noisy = np.maximum(crowd_noisy, 0)

# Background subtract
crowd_bgsub = crowd_noisy - background_level

def multi_psf_model(xy, *params):
    """
    Multiple Gaussian PSFs + shared background.
    
    Parameters: [x0_1, y0_1, amp_1, x0_2, y0_2, amp_2, ..., sigma, bg]
    """
    x, y = xy
    n_pixels = len(x)
    
    # Last two parameters are shared
    sigma_shared = params[-2]
    background = params[-1]
    
    # Number of stars
    n_stars = (len(params) - 2) // 3
    
    # Initialize model
    model = np.full(n_pixels, background)
    
    # Add each star
    for i in range(n_stars):
        x0 = params[3*i]
        y0 = params[3*i + 1]
        amp = params[3*i + 2]
        
        r_sq = (x - x0)**2 + (y - y0)**2
        model += amp * np.exp(-r_sq / (2 * sigma_shared**2))
    
    return model

# Prepare data - flatten coordinates and image
x_flat_c = x_grid_c.ravel()
y_flat_c = y_grid_c.ravel()
data_flat_c = crowd_bgsub.ravel()

# Initial guesses: [x, y, amp for each star, then shared sigma and bg]
p0_multi = []
for i in range(n_crowd):
    p0_multi.extend([crowd_x[i], crowd_y[i], crowd_amps[i]])
p0_multi.extend([psf_sigma, 0.0])  # shared sigma and background

# Fit all sources at once
popt_multi, pcov_multi = optimize.curve_fit(
    multi_psf_model,
    (x_flat_c, y_flat_c),
    data_flat_c,
    p0=p0_multi,
    maxfev=10000
)

# Extract results
fitted_x = [popt_multi[3*i] for i in range(n_crowd)]
fitted_y = [popt_multi[3*i + 1] for i in range(n_crowd)]
fitted_amps = [popt_multi[3*i + 2] for i in range(n_crowd)]
fitted_sigma = popt_multi[-2]

# Generate model
model_multi = multi_psf_model((x_flat_c, y_flat_c), *popt_multi).reshape(crowded_size, crowded_size)
residual_multi = crowd_bgsub - model_multi

# Print results
print(f"\nMulti-Source Fitting Results:")
print(f"{'Star':<6} {'True X':<10} {'Fitted X':<10} {'True Y':<10} {'Fitted Y':<10} {'True Amp':<12} {'Fitted Amp':<12}")
for i in range(n_crowd):
    print(f"{i+1:<6} {crowd_x[i]:<10.2f} {fitted_x[i]:<10.2f} {crowd_y[i]:<10.2f} {fitted_y[i]:<10.2f} {crowd_amps[i]:<12.0f} {fitted_amps[i]:<12.0f}")
print(f"\nFitted PSF sigma: {fitted_sigma:.3f} pixels (true: {psf_sigma:.3f})")
print(f"Residual RMS: {np.std(residual_multi):.2f} counts")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Data
im1 = axes[0].imshow(crowd_bgsub, cmap='hot', origin='lower', interpolation='nearest')
axes[0].scatter(crowd_x, crowd_y, c='cyan', s=100, marker='x',
               linewidths=2, label='True')
axes[0].set_title('Data (Crowded Field)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('X [pixels]')
axes[0].set_ylabel('Y [pixels]')
axes[0].legend()
plt.colorbar(im1, ax=axes[0], label='Counts')

# Model
im2 = axes[1].imshow(model_multi, cmap='hot', origin='lower', interpolation='nearest')
axes[1].scatter(fitted_x, fitted_y, c='lime', s=100, marker='+',
               linewidths=2, label='Fitted')
axes[1].set_title('Multi-Source Model', fontsize=12, fontweight='bold')
axes[1].set_xlabel('X [pixels]')
axes[1].set_ylabel('Y [pixels]')
axes[1].legend()
plt.colorbar(im2, ax=axes[1], label='Counts')

# Residuals
vmax = max(abs(residual_multi.min()), abs(residual_multi.max()))
im3 = axes[2].imshow(residual_multi, cmap='RdBu_r', origin='lower',
                    interpolation='nearest', vmin=-vmax, vmax=vmax)
axes[2].set_title(f'Residuals\nRMS = {np.std(residual_multi):.2f}', 
                 fontsize=12, fontweight='bold')
axes[2].set_xlabel('X [pixels]')
axes[2].set_ylabel('Y [pixels]')
plt.colorbar(im3, ax=axes[2], label='Counts')

plt.tight_layout()
plt.show()

## Solutions to In-Class Exercises

### Section 1 Solution

In [ ]:
r_earth = 6371 * u.km
distance = (10 * u.pc).to(u.km)
theta_rad = (2 * r_earth / distance).decompose()
theta_arcsec = (theta_rad * u.radian).to(u.arcsec)
# Result: ~0.0087". This is SMALLER than JWST's resolution (0.03").

### Section 2 Solution

In [ ]:
sigma = fwhm_seeing / 2.355
# Result: ~0.51 arcseconds

### Section 3 Solution

In [ ]:
electrons = 5000 * 1.5 # 7500 e-
noise_e = np.sqrt(7500) # 86.6 e-
noise_adu = noise_e / 1.5 # 57.7 ADU

### Section 4 Solution

In [ ]:
## Mean will be ~875 (heavily biased by the 5000). 
# Median will be 50.0 (the 5000 is ignored as an outlier).

### Section 5 Solution

In [ ]:
net_flux = 8500 - (50 * 20) # 7500 ADU

### Section 6 Solution

In [ ]:
# The fit might get stuck in a "local minimum" or fail to converge 
# because the initial guess (100, 100) is too far from the actual 
# signal (12, 15).